# Physics formulas for q95 and betaN

Measures how much of the two submitted scalars a physics formula captures on DIII-D, in two
stages: with the true boundary shape read from `efit_psirz`, then with the shape read from the
`hybrid_ridge` predicted maps, which is the situation at test time. The baseline predicts the
scalars with ridges on the raw inputs (pooled R2 0.703 for q95 and 0.934 for betaN on the
870 validation shots, `reports/baseline`). The scalar route under test is a formula times a
learned correction on quantities without units, so that it can carry over to MAST.

Formulas and shape extraction live in `fusion_eq.scalars`; the shapes come from the scorer's
own boundary and shape functions, so kappa and the triangularities are the scorer's quantities.
Section 6 adds a direct calculation of q95 from the map, the field-line integral on the
95 percent flux surface, with no fitted constant.
Cells are run manually, top to bottom. Inputs come from the frame table, the models of the
baseline comparison on the working lists (`scripts/compare_baselines.py --working`), the Green's
table and the raw DIII-D training files. Pooled R2 is one R2 over all frames, as the scorer
computes it, unclipped.

Parameters are in the next cell: the frame stride, the toroidal-field turn count, the pressure
summary and the label filter. Sections 1 and 2 leave the filter decision open: the frame cost of
every candidate rule is shown, and `FILTER` applies none unless set.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fusion_eq import baseline as bl, frame_table as ft, models, scalars as sc, split as sp, targets
from fusion_eq.coil_flux import GreenTable, adopted_scales
from fusion_eq.machines import machine_config
from fusion_eq.predictors import model_files

DATA_ROOT = Path("../../data/raw/data")
TABLE_DIR = Path("../../data/processed/frame_table")
GREEN_DIR = Path("../../data/processed/green")
CACHE_DIR = Path("../../data/processed/baseline")      # fitted models of compare_baselines.py
SHAPE_DIR = Path("../../data/processed/scalars")        # cached shape tables (rebuilt when absent)
REPORT_DIR = Path("../reports/scalars")
SPLIT_PATH = Path("../splits/split_v1.json")

WORKING = True              # working lists (300 training, 100 validation shots) or the full lists
FRAME_STRIDE = 4            # every FRAME_STRIDE-th frame of each shot, in labels, inputs and maps alike
WORKERS = None              # processes for the boundary extraction; None = CPU count minus two
N_TURNS = 144               # toroidal field coil turns, DIII-D, provisional (to verify against a published source)
PRESSURE_COLUMN = sc.DEFAULT_PRESSURE_COLUMN   # core electron pressure summary for betaN, kPa
# Label filter. Both labels finite is always required. Each candidate below is counted in section 1
# and removed from the training and validation frames only when set to True.
FILTER = {"tf_under_threshold": False, "ip_under_fraction_of_peak": False,
          "q95_outside_band": False, "betan_outside_band": False, "excluded_shots": False}
RULE_PARAMS = dict(tf_min_ka=50.0, ip_over_peak_min=0.1, q95_band=(2.91, 11.76), betan_band=(0.08, 3.31),
                   excluded_shots=("5df1630aa7", "955c35c962"))

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.precision", 4)
plt.rcParams["figure.dpi"] = 100
REPORT_DIR.mkdir(parents=True, exist_ok=True)
SHAPE_DIR.mkdir(parents=True, exist_ok=True)

CFG = machine_config("DIII-D")
TABLE = GreenTable.load(GREEN_DIR / "green_d3d.npz")
SCALES = adopted_scales("DIII-D", TABLE.columns)
GRID_R, GRID_Z = TABLE.grid_r, TABLE.grid_z

split = sp.read_split(SPLIT_PATH)
TRAIN_IDS = sorted(split["working_train" if WORKING else "train"])
VAL_IDS = sorted(split["working_validation" if WORKING else "validation"])
TRAIN_HASH = bl.hash_of_ids(TRAIN_IDS)
FILES = model_files(CACHE_DIR, TRAIN_IDS)
missing = [str(p) for k, p in FILES.items() if k != "ridge_raw" and not p.is_file()]
assert not missing, f"fitted models missing, run scripts/compare_baselines.py{' --working' if WORKING else ''}: {missing}"
print(f"{len(TRAIN_IDS)} training and {len(VAL_IDS)} validation shots ({'working' if WORKING else 'full'} lists), "
      f"stride {FRAME_STRIDE}, models of training list {TRAIN_HASH}")

In [ ]:
# Frame rows: the baseline inputs, the flags and the two labels, every FRAME_STRIDE-th frame of each shot.
COLUMNS = [*bl.input_columns(), "n_frames", "ip_over_peak", "ip_direction", "tf__mean",
           "ts_core_pe_peak__interp", "ts_core_pe_mean__interp", "efit_q95", "efit_beta_n"]


def strided_rows(shot_ids):
    rows = bl.frames_of_shots(shot_ids, COLUMNS, TABLE_DIR)
    rows["shot"] = rows["shot"].astype(str)
    rows = rows.sort_values(["shot", "frame"]).groupby("shot", sort=False).apply(lambda g: g.iloc[::FRAME_STRIDE], include_groups=False)
    return rows.reset_index(level=0).reset_index(drop=True)


t0 = time.time()
TRAIN = strided_rows(TRAIN_IDS)
VAL = strided_rows(VAL_IDS)
print(f"{len(TRAIN)} training and {len(VAL)} validation frames at stride {FRAME_STRIDE}, {time.time() - t0:.0f} s")
print(f"validation shots by Ip direction: {VAL.groupby('shot')['ip_direction'].first().value_counts().to_dict()}")

# The fitted models of the comparison: hybrid_ridge for the maps, ridge_signed for the baseline scalars.
HYBRID = bl.PcaRidge.load(FILES["hybrid_ridge"])
RIDGE_SIGNED = bl.PcaRidge.load(FILES["ridge_signed"])
with np.load(FILES["mean_shape"], allow_pickle=False) as f:
    MEAN_SHAPE, N_MEAN = f["mean"], int(f["n"])
# ModelSet.baselines()["hybrid_ridge"] reads the table, the scales, hybrid and ridge_signed only.
MODEL_SET = models.ModelSet(table=TABLE, scales=SCALES, basis_psi=RIDGE_SIGNED.basis, basis_plasma=HYBRID.basis,
                            ridge_raw=RIDGE_SIGNED, ridge_signed=RIDGE_SIGNED, hybrid=HYBRID, mean_shape=MEAN_SHAPE, n_mean_frames=N_MEAN)
HYBRID_MODEL = MODEL_SET.baselines()["hybrid_ridge"]
print(f"hybrid_ridge: {HYBRID.basis.n_components} components, {HYBRID.n_train_frames} training frames; coil model {SCALES.source}")

## 1. The label filter

Every section below uses frames with both labels finite. The table counts, for the training and
validation lists, the frames and shots each candidate rule would remove in addition: toroidal
field under 50 kA in magnitude (finding F21 and open item 8: two shots carry an implausible field
signal with q95 below 1), Ip under 0.1 of the shot's peak (F44, ramp-down frames), q95 or betaN
outside the 1st to 99th percentile band of F43 (q95 2.91 to 11.76, betaN 0.08 to 3.31), and the
two shots of the Part A follow-up (`5df1630aa7`, `955c35c962`, in the full training list but
not in the working lists). The rules are not exclusive; the last row is their union. Nothing is
decided here: `FILTER` in the parameter cell selects the rules to apply, and the kept frames of
both lists are reported.

In [ ]:
def rule_costs(frames, name):
    rules = sc.candidate_rules(frames, **RULE_PARAMS)
    valid = ~rules["invalid_label"]
    rows = [{"rule": "invalid_label", f"{name} frames": int(rules["invalid_label"].sum()),
             f"{name} shots": int(frames.loc[rules["invalid_label"], "shot"].nunique())}]
    union = np.zeros(len(frames), dtype=bool)
    for rule, mask in rules.items():
        if rule == "invalid_label":
            continue
        extra = mask & valid
        union |= extra
        rows.append({"rule": rule, f"{name} frames": int(extra.sum()), f"{name} shots": int(frames.loc[extra, "shot"].nunique())})
    rows.append({"rule": "union of candidates", f"{name} frames": int(union.sum()), f"{name} shots": int(frames.loc[union, "shot"].nunique())})
    rows.append({"rule": "valid-label frames", f"{name} frames": int(valid.sum()), f"{name} shots": int(frames.loc[valid, "shot"].nunique())})
    return pd.DataFrame(rows).set_index("rule"), rules


costs_train, RULES_TRAIN = rule_costs(TRAIN, "train")
costs_val, RULES_VAL = rule_costs(VAL, "val")
COSTS = costs_train.join(costs_val)
COSTS.to_csv(REPORT_DIR / "label_filter_costs.csv")
COSTS

In [ ]:
def keep_mask(rules):
    keep = ~rules["invalid_label"]
    for rule, on in FILTER.items():
        if on:
            keep &= ~rules[rule]
    return keep


TRAIN["keep"] = keep_mask(RULES_TRAIN)
VAL["keep"] = keep_mask(RULES_VAL)
applied = [r for r, on in FILTER.items() if on] or ["none"]
print(f"filter applied: {', '.join(applied)}")
print(f"kept: {int(TRAIN['keep'].sum())} of {len(TRAIN)} training frames, {int(VAL['keep'].sum())} of {len(VAL)} validation frames")

## 2. Toroidal field constant

The boundary shape of every kept frame is first extracted from the true map with the scorer's
functions (`scalars.shape_from_map`, cached under `SHAPE_DIR`; the first run takes a few minutes
per hundred shots). The vacuum field at the geometric major radius is
`Bt = 2e-7 N |I_tf| / R_geo` with `N = 144` turns, provisional. The multiplier `c` that best fits
the labels with the q95 formula on the training frames is then the least-squares factor; since
q95 is linear in Bt, the turn count the labels imply is `144 c`. Both are reported, with the
validation-side factor for comparison. A factor far from 1 means either the turn count or the
formula's shape factor is off; the sections after this one keep `N_TURNS` and absorb the
difference in the fitted constant and the correction.

In [ ]:
def true_items(frames):
    # (shot, frame indices, strided true maps) per shot, read one shot at a time.
    for shot_id, g in frames.groupby("shot", sort=False):
        maps = ft.read_flux_maps([shot_id], "diii_d_train", DATA_ROOT)
        idx = g["frame"].to_numpy()
        yield shot_id, idx, maps[idx]


def shapes_cached(frames, tag, items_fn):
    path = SHAPE_DIR / f"shapes_{tag}_{bl.hash_of_ids(frames['shot'].unique())}_s{FRAME_STRIDE}.parquet"
    t0 = time.time()
    table = sc.cached_shape_table(path, items_fn, GRID_R, GRID_Z, "DIII-D", WORKERS)
    print(f"{tag}: {len(table)} shapes, {int(table['boundary_found'].sum())} with a boundary, "
          f"{time.time() - t0:.0f} s ({path.name})")
    return table


SHAPES_TRUE_TRAIN = shapes_cached(TRAIN, "true_train", lambda: true_items(TRAIN))
SHAPES_TRUE_VAL = shapes_cached(VAL, "true_val", lambda: true_items(VAL))
Q_TRAIN = sc.frame_quantities(TRAIN, SHAPES_TRUE_TRAIN, CFG, N_TURNS, PRESSURE_COLUMN)
Q_VAL = sc.frame_quantities(VAL, SHAPES_TRUE_VAL, CFG, N_TURNS, PRESSURE_COLUMN)
KEEP_TRAIN, KEEP_VAL = Q_TRAIN["keep"].to_numpy(), Q_VAL["keep"].to_numpy()
NEG_VAL = (Q_VAL["ip_direction"] == -1).to_numpy()
print(f"negative-Ip validation frames: {int((KEEP_VAL & NEG_VAL).sum())} in {Q_VAL.loc[KEEP_VAL & NEG_VAL, 'shot'].nunique()} shots")

In [ ]:
kept = Q_TRAIN[KEEP_TRAIN]
bt = kept["bt"]
print(f"Bt at R_geo with {N_TURNS} turns, training frames: median {bt.median():.3f} T, "
      f"5th to 95th percentile {bt.quantile(0.05):.3f} to {bt.quantile(0.95):.3f} T; R_geo median {kept['R_geo'].median():.3f} m")
C_Q95_TRAIN = sc.best_multiplier(kept["efit_q95"], kept["q95_formula"])
C_Q95_VAL = sc.best_multiplier(Q_VAL.loc[KEEP_VAL, "efit_q95"], Q_VAL.loc[KEEP_VAL, "q95_formula"])
print(f"q95 multiplier on the formula: training {C_Q95_TRAIN:.4f} (implied turns {N_TURNS * C_Q95_TRAIN:.1f}), "
      f"validation {C_Q95_VAL:.4f} (implied turns {N_TURNS * C_Q95_VAL:.1f})")
ratio = kept["efit_q95"] / kept["q95_formula"]
print(f"label over formula, training: median {ratio.median():.4f}, 5th to 95th percentile {ratio.quantile(0.05):.3f} to {ratio.quantile(0.95):.3f}")

## 3. q95, stage 1: true shapes

Pooled R2 of the formula against `efit_q95` on the kept validation frames, alone and times the
training-side constant of section 2; frames without a boundary count as the pooled mean, as the
scorer fills a missing prediction. Then the scatter of label against formula and the ratio
label over formula against the shape quantities, Ip over its peak and the Thomson peak-to-mean
ratio. A flat ratio against a quantity means the formula already carries that dependence; a trend
is what the correction of section 4 has to learn.

In [ ]:
RESULTS = []


def record(scalar, stage, method, label, pred, keep=KEEP_VAL, neg=NEG_VAL, **extra):
    y, p = np.asarray(label, dtype=np.float64), np.asarray(pred, dtype=np.float64)
    row = {"scalar": scalar, "stage": stage, "method": method, "r2_all": sc.pooled_r2(y[keep], p[keep]),
           "r2_negative_ip": sc.pooled_r2(y[keep & neg], p[keep & neg]) if (keep & neg).any() else np.nan,
           "n_frames": int(keep.sum()), "n_frames_negative_ip": int((keep & neg).sum()),
           "n_missing_pred": int((~np.isfinite(p[keep])).sum()), **extra}
    RESULTS.append(row)
    return row


ridge_val = bl.predict(RIDGE_SIGNED, targets.signed_frames(VAL, CFG))
r = record("q95", 1, "ridge baseline (inputs)", Q_VAL["efit_q95"], ridge_val.q95)
print(f"ridge_signed q95 on the same validation frames: R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f})")
r = record("q95", 1, "formula", Q_VAL["efit_q95"], Q_VAL["q95_formula"])
print(f"formula alone: R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f}); {r['n_missing_pred']} frames without a boundary")
r = record("q95", 1, "formula x constant", Q_VAL["efit_q95"], C_Q95_TRAIN * Q_VAL["q95_formula"])
print(f"formula x {C_Q95_TRAIN:.4f}: R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f})")

In [ ]:
def scatter_label_formula(q, keep, label, formula, name, constant):
    fig, ax = plt.subplots(figsize=(5.2, 5))
    k = keep & np.isfinite(q[formula])
    ax.scatter(constant * q.loc[k, formula], q.loc[k, label], s=3, alpha=0.25, c=np.where(q.loc[k, "ip_direction"] == -1, "C3", "C0"))
    lim = [0, np.nanpercentile(q.loc[k, label], 99.5) * 1.1]
    ax.plot(lim, lim, "k--", lw=0.8)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel(f"{constant:.3f} x formula"); ax.set_ylabel(label)
    ax.set_title(f"{name}, validation frames (red: negative Ip)", fontsize=10)
    return fig


def ratio_panels(q, keep, label, formula, name, quantities=("kappa", "delta", "eps", "ip_over_peak", "peak_to_mean")):
    k = keep & np.isfinite(q[formula]) & (q[formula] > 0)
    ratio = (q.loc[k, label] / q.loc[k, formula]).to_numpy()
    fig, axes = plt.subplots(1, len(quantities), figsize=(3.4 * len(quantities), 3.6), sharey=True)
    for ax, col in zip(axes, quantities):
        x = q.loc[k, col].to_numpy()
        ok = np.isfinite(x)
        ax.scatter(x[ok], ratio[ok], s=2, alpha=0.15, color="0.5")
        edges = np.nanpercentile(x[ok], np.linspace(0, 100, 13))
        centres, medians = [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            b = ok & (x >= lo) & (x <= hi)
            if b.sum() >= 20:
                centres.append(np.median(x[b])); medians.append(np.median(ratio[b]))
        ax.plot(centres, medians, "o-", color="C1", ms=4)
        ax.axhline(1.0, color="k", lw=0.6, ls="--")
        ax.set_xlabel(col)
    axes[0].set_ylabel(f"{label} / formula")
    axes[0].set_ylim(np.nanpercentile(ratio, 0.5) * 0.9, np.nanpercentile(ratio, 99.5) * 1.1)
    fig.suptitle(f"{name}: label over {formula} (orange: median per bin)", fontsize=10)
    fig.tight_layout()
    return fig


scatter_label_formula(Q_VAL, KEEP_VAL, "efit_q95", "q95_formula", "q95 stage 1", C_Q95_TRAIN).savefig(REPORT_DIR / "q95_stage1_scatter.png", bbox_inches="tight")
ratio_panels(Q_VAL, KEEP_VAL, "efit_q95", "q95_formula", "q95 stage 1").savefig(REPORT_DIR / "q95_stage1_ratio.png", bbox_inches="tight")

## 4. q95 correction

A model of the ratio label over formula, learned on the training frames from quantities
without units only: kappa, tri_top, tri_bot, eps, the Thomson peak-to-mean ratio and Ip over
its peak (`scalars.CORRECTION_FEATURES`). Two models (`scalars.CorrectionModel`): a ridge on the
standardized inputs and a small gradient-boosted model, both fitted on the log of the ratio so
the correction is a positive factor. Pooled R2 of formula times factor on the validation frames.
The ridge coefficients are on standardized inputs, so their sizes compare directly.

In [ ]:
def fit_corrections(q_train, keep_train, label, formula):
    rows = q_train[keep_train]
    ridge = sc.CorrectionModel(method="ridge").fit(rows, rows[label], rows[formula])
    gbm = sc.CorrectionModel(method="gbm").fit(rows, rows[label], rows[formula])
    return ridge, gbm


CORR_Q95 = fit_corrections(Q_TRAIN, KEEP_TRAIN, "efit_q95", "q95_formula")
for name, model in zip(("ridge correction", "gbm correction"), CORR_Q95):
    r = record("q95", 1, f"formula x {name}", Q_VAL["efit_q95"], model.predict(Q_VAL, Q_VAL["q95_formula"]))
    print(f"q95 stage 1, {name} ({model.n_train} training frames): R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f})")
print("ridge coefficients on standardized inputs (log ratio):",
      {f: round(float(c), 4) for f, c in zip(CORR_Q95[0].features, CORR_Q95[0].model.coef_)}, f"intercept {CORR_Q95[0].model.intercept_:.4f}")

## 5. q95, stage 2: predicted shapes

The situation at test time: the shapes are read from the `hybrid_ridge` predicted maps
(`models.ModelSet.baselines()["hybrid_ridge"]`, the model of the baseline comparison on the
same training list) on the validation frames, with the same scorer functions. The constant and
the corrections are refitted on the predicted shapes of the training frames, so that the
correction learns the biases of the predicted shapes as well. The stage-1 correction applied to
the predicted shapes is reported too: the difference between the two says how much of the
correction is shape bias. The agreement of the predicted and true shape quantities on the
validation frames is printed first; it is the quantity the consistency term also scores.

In [ ]:
def predicted_items(frames):
    for shot_id, g in frames.groupby("shot", sort=False):
        maps = HYBRID_MODEL.predict(g).psirz
        yield shot_id, g["frame"].to_numpy(), maps


SHAPES_PRED_TRAIN = shapes_cached(TRAIN, f"hybrid{TRAIN_HASH}_train", lambda: predicted_items(TRAIN))
SHAPES_PRED_VAL = shapes_cached(VAL, f"hybrid{TRAIN_HASH}_val", lambda: predicted_items(VAL))
P_TRAIN = sc.frame_quantities(TRAIN, SHAPES_PRED_TRAIN, CFG, N_TURNS, PRESSURE_COLUMN)
P_VAL = sc.frame_quantities(VAL, SHAPES_PRED_VAL, CFG, N_TURNS, PRESSURE_COLUMN)
agreement = {col: sc.pooled_r2(Q_VAL.loc[KEEP_VAL, col], P_VAL.loc[KEEP_VAL, col]) for col in sc.SHAPE_COLUMNS}
print("predicted against true shape quantities, pooled R2 on the kept validation frames:", {k: round(v, 4) for k, v in agreement.items()})

In [ ]:
C_Q95_PRED = sc.best_multiplier(P_TRAIN.loc[KEEP_TRAIN, "efit_q95"], P_TRAIN.loc[KEEP_TRAIN, "q95_formula"])
print(f"q95 multiplier on predicted shapes, training: {C_Q95_PRED:.4f} (true shapes {C_Q95_TRAIN:.4f})")
r = record("q95", 2, "formula", P_VAL["efit_q95"], P_VAL["q95_formula"])
print(f"formula alone: R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f}); {r['n_missing_pred']} frames without a boundary")
r = record("q95", 2, "formula x constant", P_VAL["efit_q95"], C_Q95_PRED * P_VAL["q95_formula"])
print(f"formula x {C_Q95_PRED:.4f}: R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f})")
CORR_Q95_PRED = fit_corrections(P_TRAIN, KEEP_TRAIN, "efit_q95", "q95_formula")
for name, model, stage1 in zip(("ridge correction", "gbm correction"), CORR_Q95_PRED, CORR_Q95):
    r = record("q95", 2, f"formula x {name}", P_VAL["efit_q95"], model.predict(P_VAL, P_VAL["q95_formula"]))
    r1 = record("q95", 2, f"formula x {name} (stage-1 fit)", P_VAL["efit_q95"], stage1.predict(P_VAL, P_VAL["q95_formula"]))
    print(f"q95 stage 2, {name}: R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f}); with the stage-1 fit {r1['r2_all']:.4f}")
record("q95", 2, "ridge baseline (inputs)", Q_VAL["efit_q95"], ridge_val.q95)
scatter_label_formula(P_VAL, KEEP_VAL, "efit_q95", "q95_formula", "q95 stage 2", C_Q95_PRED).savefig(REPORT_DIR / "q95_stage2_scatter.png", bbox_inches="tight")
ratio_panels(P_VAL, KEEP_VAL, "efit_q95", "q95_formula", "q95 stage 2").savefig(REPORT_DIR / "q95_stage2_ratio.png", bbox_inches="tight")

## 6. q95 calculated from the map

A direct calculation instead of a formula (`scalars.q95_from_map`): the scorer's boundary and
axis give `psi_N = (psi - psi_axis) / (psi_boundary - psi_axis)`; the closed contour at
`psi_N = 0.95` enclosing the axis is the q95 surface, and

    q95 = F / (2 pi) x closed integral of dl / (R |grad psi|),   F = R Bt = 2e-7 N |I_tf|

with the gradient by central differences sampled on the contour, psi in Wb/rad and `N_TURNS`
turns, no fitted constant. A frame with no boundary or no closed surface gets NaN; those frames
are counted and, in the filled rows of the summary, replaced by the formula times the
training-side constant of section 2. The same working subsets and frames as above; the shapes
for the ratio panels come from the cached shape tables. Stage 1 runs on the true validation maps
at `psi_N` 0.95, 0.94 and 0.96 to show the sensitivity to the surface chosen; stage 2 on the
`hybrid_ridge` predicted maps at 0.95. The calculation runs on the validation frames only, since
nothing is fitted.

In [ ]:
Q95_LEVELS = (0.95, 0.94, 0.96)
CALC_COL = sc.q95_column(0.95)


def true_items_tf(frames):
    for shot_id, g in frames.groupby("shot", sort=False):
        maps = ft.read_flux_maps([shot_id], "diii_d_train", DATA_ROOT)
        idx = g["frame"].to_numpy()
        yield shot_id, idx, maps[idx], g["tf__mean"].to_numpy(dtype=np.float64)


def predicted_items_tf(frames):
    for shot_id, g in frames.groupby("shot", sort=False):
        yield shot_id, g["frame"].to_numpy(), HYBRID_MODEL.predict(g).psirz, g["tf__mean"].to_numpy(dtype=np.float64)


def q95_cached(frames, tag, items_fn, levels):
    path = SHAPE_DIR / f"q95calc_{tag}_{bl.hash_of_ids(frames['shot'].unique())}_s{FRAME_STRIDE}_n{N_TURNS}.parquet"
    t0 = time.time()
    table = sc.cached_q95_table(path, items_fn, GRID_R, GRID_Z, N_TURNS, "DIII-D", levels, WORKERS)
    print(f"{tag}: {len(table)} frames, failures per level "
          f"{ {c: int(table[c].isna().sum()) for c in table.columns if c.startswith('q95_calc')} }, {time.time() - t0:.0f} s ({path.name})")
    return table


CALC_TRUE_VAL = q95_cached(VAL, "true_val", lambda: true_items_tf(VAL), Q95_LEVELS)
CALC_PRED_VAL = q95_cached(VAL, f"hybrid{TRAIN_HASH}_val", lambda: predicted_items_tf(VAL), (0.95,))
Q_VAL = Q_VAL.drop(columns=[c for c in Q_VAL.columns if c.startswith("q95_calc")]).merge(CALC_TRUE_VAL, on=["shot", "frame"], how="left", validate="one_to_one")
P_VAL = P_VAL.drop(columns=[c for c in P_VAL.columns if c.startswith("q95_calc")]).merge(CALC_PRED_VAL, on=["shot", "frame"], how="left", validate="one_to_one")

In [ ]:
def calc_results(q, stage, constant, name):
    calc = q[CALC_COL].to_numpy(dtype=np.float64)
    failed = KEEP_VAL & ~np.isfinite(calc)
    filled = np.where(np.isfinite(calc), calc, constant * q["q95_formula"].to_numpy(dtype=np.float64))
    r = record("q95", stage, "calculated from map (psi_N 0.95)", q["efit_q95"], calc, n_calc_failed=int(failed.sum()))
    r_f = record("q95", stage, "calculated from map, failures filled with formula x constant", q["efit_q95"], filled, n_calc_failed=int(failed.sum()))
    k = KEEP_VAL & np.isfinite(calc) & (calc > 0)
    median = float(np.median(q.loc[k, "efit_q95"] / q.loc[k, CALC_COL]))
    print(f"{name}: R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f}), {int(failed.sum())} of {int(KEEP_VAL.sum())} kept frames failed "
          f"(counted as the pooled mean); filled with {constant:.4f} x formula: R2 {r_f['r2_all']:.4f} (negative Ip {r_f['r2_negative_ip']:.4f}); "
          f"median label / calculated {median:.4f}")
    return median, int(failed.sum())


CALC_MEDIANS, CALC_FAILED_TRUE = {}, None
CALC_MEDIANS["0.95"], CALC_FAILED_TRUE = calc_results(Q_VAL, 1, C_Q95_TRAIN, "q95 from the true map, psi_N 0.95")
scatter_label_formula(Q_VAL, KEEP_VAL, "efit_q95", CALC_COL, "q95 calculated, stage 1", 1.0).savefig(REPORT_DIR / "q95_calc_stage1_scatter.png", bbox_inches="tight")
ratio_panels(Q_VAL, KEEP_VAL, "efit_q95", CALC_COL, "q95 calculated, stage 1", ("kappa", "delta", "eps", "ip_over_peak")).savefig(REPORT_DIR / "q95_calc_stage1_ratio.png", bbox_inches="tight")

In [ ]:
# Sensitivity to the surface: 0.94 and 0.96 against 0.95 on the true validation maps.
for level in (0.94, 0.96):
    col = sc.q95_column(level)
    calc = Q_VAL[col].to_numpy(dtype=np.float64)
    r = record("q95", 1, f"calculated from map (psi_N {level:.2f})", Q_VAL["efit_q95"], calc, n_calc_failed=int((KEEP_VAL & ~np.isfinite(calc)).sum()))
    k = KEEP_VAL & np.isfinite(calc) & np.isfinite(Q_VAL[CALC_COL].to_numpy()) & (calc > 0)
    CALC_MEDIANS[f"{level:.2f}"] = float(np.median(Q_VAL.loc[k, "efit_q95"] / Q_VAL.loc[k, col]))
    relative = np.median(Q_VAL.loc[k, col] / Q_VAL.loc[k, CALC_COL])
    print(f"psi_N {level:.2f}: R2 {r['r2_all']:.4f}, median label / calculated {CALC_MEDIANS[f'{level:.2f}']:.4f}, "
          f"median q({level:.2f}) / q(0.95) {relative:.4f}, {r['n_calc_failed']} failures")

In [ ]:
CALC_MEDIAN_PRED, CALC_FAILED_PRED = calc_results(P_VAL, 2, C_Q95_PRED, "q95 from the hybrid_ridge map, psi_N 0.95")
scatter_label_formula(P_VAL, KEEP_VAL, "efit_q95", CALC_COL, "q95 calculated, stage 2", 1.0).savefig(REPORT_DIR / "q95_calc_stage2_scatter.png", bbox_inches="tight")
ratio_panels(P_VAL, KEEP_VAL, "efit_q95", CALC_COL, "q95 calculated, stage 2", ("kappa", "delta", "eps", "ip_over_peak")).savefig(REPORT_DIR / "q95_calc_stage2_ratio.png", bbox_inches="tight")

## 7. betaN

Sections 3 to 5 repeated with `betaN = 100 (2 mu0 p / Bt^2) a Bt / Ip`, with `p` the core
Thomson electron pressure summary (`PRESSURE_COLUMN`, kPa in the frame table, converted to Pa).
Thomson gives the electron pressure only, so the label should exceed the formula by a roughly
constant factor (total over electron pressure); that factor is the fitted constant. Frames
without a valid core profile have no pressure and count as the pooled mean. The last cell scores
the variant that predicts `betaN x |Ip|` and divides by `|Ip|` afterwards. On the formula route
the variant is algebraically the same model (the ratio label over formula does not change when
both sides are multiplied by Ip), so it is scored on the learned baseline instead: a ridge on
the baseline inputs predicting `betaN x |Ip|` in MA, divided by `|Ip|`, against the direct ridge.

In [ ]:
C_BN_TRAIN = sc.best_multiplier(Q_TRAIN.loc[KEEP_TRAIN, "efit_beta_n"], Q_TRAIN.loc[KEEP_TRAIN, "betan_formula"])
C_BN_VAL = sc.best_multiplier(Q_VAL.loc[KEEP_VAL, "efit_beta_n"], Q_VAL.loc[KEEP_VAL, "betan_formula"])
bn_ratio = Q_TRAIN.loc[KEEP_TRAIN, "efit_beta_n"] / Q_TRAIN.loc[KEEP_TRAIN, "betan_formula"]
print(f"betaN constant factor (label over electron-pressure formula): training {C_BN_TRAIN:.3f}, validation {C_BN_VAL:.3f}; "
      f"ratio median {bn_ratio.median():.3f}, 5th to 95th percentile {bn_ratio.quantile(0.05):.3f} to {bn_ratio.quantile(0.95):.3f}")
print(f"frames without a pressure value: training {int((~np.isfinite(Q_TRAIN.loc[KEEP_TRAIN, 'p_pa'])).sum())}, "
      f"validation {int((~np.isfinite(Q_VAL.loc[KEEP_VAL, 'p_pa'])).sum())}")

r = record("betaN", 1, "ridge baseline (inputs)", Q_VAL["efit_beta_n"], ridge_val.betaN)
print(f"ridge_signed betaN on the same validation frames: R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f})")
r = record("betaN", 1, "formula", Q_VAL["efit_beta_n"], Q_VAL["betan_formula"])
print(f"formula alone: R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f}); {r['n_missing_pred']} frames without boundary or pressure")
r = record("betaN", 1, "formula x constant", Q_VAL["efit_beta_n"], C_BN_TRAIN * Q_VAL["betan_formula"])
print(f"formula x {C_BN_TRAIN:.3f}: R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f})")
CORR_BN = fit_corrections(Q_TRAIN, KEEP_TRAIN, "efit_beta_n", "betan_formula")
for name, model in zip(("ridge correction", "gbm correction"), CORR_BN):
    r = record("betaN", 1, f"formula x {name}", Q_VAL["efit_beta_n"], model.predict(Q_VAL, Q_VAL["betan_formula"]))
    print(f"betaN stage 1, {name}: R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f})")
scatter_label_formula(Q_VAL, KEEP_VAL, "efit_beta_n", "betan_formula", "betaN stage 1", C_BN_TRAIN).savefig(REPORT_DIR / "betan_stage1_scatter.png", bbox_inches="tight")
ratio_panels(Q_VAL, KEEP_VAL, "efit_beta_n", "betan_formula", "betaN stage 1").savefig(REPORT_DIR / "betan_stage1_ratio.png", bbox_inches="tight")

In [ ]:
C_BN_PRED = sc.best_multiplier(P_TRAIN.loc[KEEP_TRAIN, "efit_beta_n"], P_TRAIN.loc[KEEP_TRAIN, "betan_formula"])
r = record("betaN", 2, "formula", P_VAL["efit_beta_n"], P_VAL["betan_formula"])
print(f"betaN stage 2, formula alone: R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f})")
r = record("betaN", 2, "formula x constant", P_VAL["efit_beta_n"], C_BN_PRED * P_VAL["betan_formula"])
print(f"formula x {C_BN_PRED:.3f} (predicted-shape constant): R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f})")
CORR_BN_PRED = fit_corrections(P_TRAIN, KEEP_TRAIN, "efit_beta_n", "betan_formula")
for name, model, stage1 in zip(("ridge correction", "gbm correction"), CORR_BN_PRED, CORR_BN):
    r = record("betaN", 2, f"formula x {name}", P_VAL["efit_beta_n"], model.predict(P_VAL, P_VAL["betan_formula"]))
    r1 = record("betaN", 2, f"formula x {name} (stage-1 fit)", P_VAL["efit_beta_n"], stage1.predict(P_VAL, P_VAL["betan_formula"]))
    print(f"betaN stage 2, {name}: R2 {r['r2_all']:.4f} (negative Ip {r['r2_negative_ip']:.4f}); with the stage-1 fit {r1['r2_all']:.4f}")
record("betaN", 2, "ridge baseline (inputs)", Q_VAL["efit_beta_n"], ridge_val.betaN)
scatter_label_formula(P_VAL, KEEP_VAL, "efit_beta_n", "betan_formula", "betaN stage 2", C_BN_PRED).savefig(REPORT_DIR / "betan_stage2_scatter.png", bbox_inches="tight")

In [ ]:
# Variant on the learned baseline: a ridge on the baseline inputs predicting betaN x |Ip| (MA), divided by |Ip| afterwards.
x_train, complete_train = RIDGE_SIGNED.scaler.transform(targets.signed_frames(TRAIN, CFG))
x_val, _ = RIDGE_SIGNED.scaler.transform(targets.signed_frames(VAL, CFG))
ip_train = np.abs(Q_TRAIN["ip_ma"].to_numpy())
ip_val = np.abs(Q_VAL["ip_ma"].to_numpy())
y_train = Q_TRAIN["efit_beta_n"].to_numpy() * ip_train
fit = complete_train & KEEP_TRAIN & np.isfinite(y_train)
w, b = bl.ridge_fit(x_train[fit], y_train[fit], RIDGE_SIGNED.params.alpha)
with np.errstate(divide="ignore", invalid="ignore"):
    betan_product = (x_val @ w[:, 0] + b[0]) / ip_val
# The direct ridge refitted on the same frames, so that the two differ in the target alone.
y_direct = Q_TRAIN["efit_beta_n"].to_numpy()
w2, b2 = bl.ridge_fit(x_train[fit], y_direct[fit], RIDGE_SIGNED.params.alpha)
betan_direct = x_val @ w2[:, 0] + b2[0]
for stage in (1, 2):
    r = record("betaN", stage, "ridge on inputs, betaN x Ip then divided", Q_VAL["efit_beta_n"], betan_product)
    r2 = record("betaN", stage, "ridge on inputs, refit on these frames", Q_VAL["efit_beta_n"], betan_direct)
print(f"ridge on inputs refit on the kept training frames: R2 {r2['r2_all']:.4f}; predicting betaN x Ip then dividing: {r['r2_all']:.4f} "
      f"(negative Ip {r2['r2_negative_ip']:.4f} and {r['r2_negative_ip']:.4f})")

## 8. Summary

Pooled R2 on the kept validation frames, all frames and negative-Ip shots, for the ridge baseline
of the comparison on the same frames, the formula alone, the formula times the training-side
constant and the formula times each correction, at stage 1 (true shapes) and stage 2 (shapes
from the `hybrid_ridge` maps, constants and corrections refitted on predicted training shapes).
Written to `reports/scalars/summary.parquet` and `summary.md` with the run parameters.

In [ ]:
SUMMARY = pd.DataFrame(RESULTS).drop_duplicates(subset=["scalar", "stage", "method"], keep="last")
ORDER = ["ridge baseline (inputs)", "formula", "formula x constant", "formula x ridge correction", "formula x gbm correction",
         "calculated from map (psi_N 0.95)", "calculated from map, failures filled with formula x constant",
         "calculated from map (psi_N 0.94)", "calculated from map (psi_N 0.96)",
         "formula x ridge correction (stage-1 fit)", "formula x gbm correction (stage-1 fit)",
         "ridge on inputs, refit on these frames", "ridge on inputs, betaN x Ip then divided"]
SUMMARY["method"] = pd.Categorical(SUMMARY["method"], [m for m in ORDER if m in set(SUMMARY["method"])] + sorted(set(SUMMARY["method"]) - set(ORDER)))
SUMMARY = SUMMARY.sort_values(["scalar", "stage", "method"]).reset_index(drop=True)
SUMMARY.to_parquet(REPORT_DIR / "summary.parquet", index=False)
params = {"working": WORKING, "frame_stride": FRAME_STRIDE, "n_turns": N_TURNS, "pressure_column": PRESSURE_COLUMN, "filter": FILTER,
          "rule_params": {k: list(v) if isinstance(v, tuple) else v for k, v in RULE_PARAMS.items()}, "train_hash": TRAIN_HASH,
          "n_train_shots": len(TRAIN_IDS), "n_val_shots": len(VAL_IDS), "kept_train_frames": int(KEEP_TRAIN.sum()), "kept_val_frames": int(KEEP_VAL.sum()),
          "q95_constant": {"true_train": C_Q95_TRAIN, "true_val": C_Q95_VAL, "pred_train": C_Q95_PRED, "implied_turns_true_train": N_TURNS * C_Q95_TRAIN},
          "betan_constant": {"true_train": C_BN_TRAIN, "true_val": C_BN_VAL, "pred_train": C_BN_PRED}, "shape_agreement_r2": agreement,
          "q95_calc": {"levels": list(Q95_LEVELS), "median_label_over_calc_true": CALC_MEDIANS, "median_label_over_calc_pred": CALC_MEDIAN_PRED,
                       "failed_true_val": CALC_FAILED_TRUE, "failed_pred_val": CALC_FAILED_PRED}}
(REPORT_DIR / "summary.json").write_text(json.dumps(params, indent=2, default=float) + "\n")
lines = [f"# Scalar formulas, {'working' if WORKING else 'full'} lists of `{SPLIT_PATH.name}`, stride {FRAME_STRIDE}", "",
         f"{len(TRAIN_IDS)} training and {len(VAL_IDS)} validation shots; kept frames {int(KEEP_TRAIN.sum())} and {int(KEEP_VAL.sum())}; "
         f"filter: {', '.join(applied)}; N_TURNS {N_TURNS}; pressure `{PRESSURE_COLUMN}`.", ""]
for (scalar, stage), g in SUMMARY.groupby(["scalar", "stage"], observed=True):
    lines += [f"## {scalar}, stage {stage}", "", "| Method | R2 all | R2 negative Ip | frames | missing pred | calc failed |", "| --- | --- | --- | --- | --- | --- |"]
    lines += [f"| {r.method} | {r.r2_all:.4f} | {r.r2_negative_ip:.4f} | {r.n_frames} | {r.n_missing_pred} | "
              f"{'' if pd.isna(getattr(r, 'n_calc_failed', np.nan)) else int(r.n_calc_failed)} |" for r in g.itertuples()]
    lines.append("")
(REPORT_DIR / "summary.md").write_text("\n".join(lines))
SUMMARY

## 9. Findings

Run of 11 October 2026 on the working lists of `split_v1.json` (300 training and 100
validation shots, every 4th frame, 16,423 and 5,551 frames with both labels finite, no filter
applied, 144 turns, core `pe_mean`). Numbers from `reports/scalars/summary.md` and
`summary.json`; the same findings are F73 to F77 in `docs/findings.md`.

### q95

- **Formula** (section 2 to 5). Least-squares multiplier on the formula with the true shapes
  0.771 on training (111 turns if read as a turn count), 0.785 on validation. Stage 1:
  formula alone R2 0.307, times the constant 0.948, ridge correction 0.967, gradient-boosted
  correction 0.978; the ridge baseline on the inputs 0.736 on the same frames. Stage 2
  (shapes from the hybrid-ridge maps, refitted): 0.326, 0.937, 0.950, 0.969; the stage-1
  corrections on the predicted shapes 0.951 and 0.961. Negative-Ip frames at stage 1:
  constant 0.986, gradient-boosted 0.983, ridge baseline 0.744. Label over formula drifts from
  0.83 at eps 0.33 to 0.76 at eps 0.37 and from 0.83 at delta 0.3 to 0.77 at delta 0.8; flat in
  kappa and in the Thomson peak-to-mean ratio; 0.77 at half the peak current to 0.80 at the
  peak. Reading: the aspect-ratio and triangularity factors over-correct on DIII-D shapes.
- **Calculated from the map** (section 6). No failure on 5,551 frames at any level. Stage 1:
  R2 0.9685 (negative Ip 0.9942), median label over calculated 0.9974 at psi_N 0.95; 1.0391
  at 0.94 (R2 0.9660) and 0.9523 at 0.96 (R2 0.9293). Stage 2: R2 0.9388 (negative Ip
  0.9722), median 0.9996. The 144 turns are confirmed to 0.3 percent at the median; the
  111 turns implied by the formula was its shape factor. At stage 2 the calculation sits
  below the corrected formula because it inherits the predicted map's shape errors.
- **Shape agreement** (section 5): predicted against true shapes, pooled R2 R_geo 0.682,
  a 0.627, kappa 0.557, tri_top 0.921, tri_bot 0.903, eps 0.747.

### betaN

Constant factor label over the electron-pressure formula 3.67 on training, 3.50 on
validation. Stage 1: formula alone -0.840, times the constant 0.790, ridge correction 0.813,
gradient-boosted correction 0.851; the ridge baseline 0.940 (0.941 refitted on these frames).
Stage 2: 0.800, 0.850, 0.899; stage-1 corrections on the predicted shapes 0.824 and 0.840.
Negative-Ip frames: gradient-boosted 0.797 and 0.829 against the ridge's 0.891. The ridge
predicting betaN times the absolute Ip and dividing afterwards: 0.924 against 0.941 direct.
Reading: one electron-pressure summary carries less of betaN than the full input set.

### Label filter (section 1)

No frame with an invalid label on either list. Toroidal field under 50 kA: 10 training frames
in 1 shot, none on validation; Ip under 0.1 of the peak: 1 training frame. q95 outside the
F43 band: 259 training frames (1.6 percent) in 113 shots, 112 validation frames in 44 shots;
betaN outside the band: 338 (2.1 percent) in 88 shots, 161 in 30. The two flagged shots are
not in the working lists. Union: 568 training frames (3.5 percent) in 150 shots, 263
validation frames (4.7 percent) in 53 shots. Reading: the bands remove ramp frames from half
the shots rather than whole shots.

### Carried into modelling

- q95 is calculated from the predicted map, 144 turns on DIII-D, no fitted constant; on
  MAST the same calculation without a learned correction, once the MAST turn count is
  verified from a published source.
- betaN stays formula times correction, with the correction to be given transferable coil
  features; which inputs transfer is untested.
- Training uses the lighter label filter (impossible values, field under 50 kA, Ip under 0.1
  of the peak, the two flagged shots); the percentile bands are to be checked on the full lists.
